# Newton's method

_Machine Learning — more_

**Use curvature, not just slope. Jump to the bottom of a parabola each step.**

Gradient descent only knows the slope. It takes many small, cautious steps.

     Newton's method also uses the curvature (how the slope changes).

---

This notebook is a practice scaffold for the **Newton's method** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — scikit-learn

In [ ]:
import numpy as np

# Convex cost J(t) = 3t^2 - 12t + 7. Minimum at t = 2.
J   = lambda t: 3*t**2 - 12*t + 7   # cost
Jp  = lambda t: 6*t - 12            # first derivative (slope)
Jpp = lambda t: 6.0                 # second derivative (curvature, constant)

t = 5.0                             # start anywhere
print("start: t=%.6f  J=%.6f  J'=%.6f" % (t, J(t), Jp(t)))
for i in range(1, 5):
    t = t - Jp(t) / Jpp(t)         # Newton step: slope / curvature
    print("step %d: t=%.6f  J=%.6f  J'=%.6f" % (i, t, J(t), Jp(t)))
    if abs(Jp(t)) < 1e-9:
        break
print("minimum at t=%.6f (exact = 2.0) reached in 1 step" % t)


## Visualize the data & results

_Can curvature steps land on the minimum of a real loss?_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer

# REAL data: 569 breast-cancer patients, feature 0 = mean tumor radius.
bc = load_breast_cancer()
x = bc.data[:, 0]
x = (x - x.mean()) / x.std()          # standardize the radius
y = (bc.target == 0).astype(float)    # 1 = malignant, 0 = benign

b = np.log(y.mean() / (1 - y.mean())) # fixed base log-odds intercept

def nll(w):                           # average logistic loss for slope w
    z = b + w*x
    return np.mean(np.log1p(np.exp(-(2*y - 1)*z)))
def grad(w):
    p = 1/(1 + np.exp(-(b + w*x)))
    return np.mean((p - y)*x)
def hess(w):                          # curvature of the loss
    p = 1/(1 + np.exp(-(b + w*x)))
    return np.mean(p*(1 - p)*x*x)

w = 0.0
iters = [w]
for _ in range(6):
    w = w - grad(w)/hess(w)           # Newton step: slope / curvature
    iters.append(w)
    if np.isclose(grad(w), 0.0):     # gradient essentially zero, stop
        break

grid = np.linspace(-1, 6, 29)
plt.plot(grid, [nll(v) for v in grid], color='#4ea1ff', label='logistic loss NLL(w)')
plt.plot(iters, [nll(v) for v in iters], 'o-', color='#c89bff',
         label='Newton iterates (start 0 to min 3.64)')
plt.xlabel('slope weight w (on mean tumor radius)')
plt.ylabel('average logistic loss (NLL)')
plt.title('Newton iterates slide down the breast-cancer logistic loss')
plt.legend(); plt.show()
